In [1]:
import torch 
# Check if CUDA is available
print("CUDA available:", torch.cuda.is_available())

# Get the number of available GPUs
print("Number of GPUs:", torch.cuda.device_count())

# Display information about each GPU
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
        
    # Show current GPU
    print("Current GPU:", torch.cuda.current_device())
    
    # Show memory info for the current device
    print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")



CUDA available: True
Number of GPUs: 1
GPU 0: NVIDIA GeForce RTX 5060 Ti
Current GPU: 0
Memory allocated: 0.00 MB
Memory reserved: 0.00 MB


In [6]:
import os
import time
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm.auto import tqdm

def train_model(device, epochs=20, batch_size=256, learning_rate=1e-2):
    print(f"\n=== Training on {device} ===")
    
    # Model definition
    class LetterNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([
                nn.Linear(16, 512),
                nn.Linear(512, 512),
                nn.Linear(512, 512),
                nn.Linear(512, 256),
                nn.Linear(256, 256)
            ])
            self.output = nn.Linear(256, 26)
            self.activation = F.relu
            self.dropout = nn.Dropout(0.2)
            
        def forward(self, x):
            for layer in self.layers:
                x = self.activation(layer(x))
                x = self.dropout(x)
            return self.output(x)
    
    # Data preparation
    print("Loading and preparing data...")
    X, y = fetch_openml('letter', version=1, return_X_y=True, as_frame=False)
    X = StandardScaler().fit_transform(X.astype('float32'))
    y = LabelEncoder().fit_transform(y)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=1
    )
    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    test_ds = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))
    
    # Training setup
    model = LetterNet().to(device)
    print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = nn.CrossEntropyLoss(reduction='sum')
    
    # Training loop
    print("Starting training...")
    start_time = time.time()
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            outputs = model(xb)
            loss = loss_fn(outputs, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        correct = total = 0
        test_loss = 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                outputs = model(xb)
                test_loss += loss_fn(outputs, yb).item()
                preds = outputs.argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)
        
        accuracy = correct / total
        avg_train_loss = train_loss / len(X_tr)
        avg_test_loss = test_loss / len(X_te)
        
        print(f"Epoch {epoch+1}/{epochs} - "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Test Loss: {avg_test_loss:.4f}, "
              f"Accuracy: {accuracy:.4f}")
    
    total_time = time.time() - start_time
    print(f"Training completed in {total_time:.2f} seconds")
    print(f"Final accuracy: {accuracy:.4f}")
    
    if torch.cuda.is_available() and device.type == 'cuda':
        print(f"Memory allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Memory reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")
    
    return total_time, accuracy

# Compare CPU vs GPU
cpu_device = torch.device("cpu")
gpu_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Verify GPU availability
if gpu_device.type == "cuda":
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU available, comparison will use CPU for both runs")

# Run on CPU
cpu_time, cpu_accuracy = train_model(cpu_device, epochs=20)

# Run on GPU if available
if gpu_device.type == "cuda":
    # Clear GPU memory before second run
    torch.cuda.empty_cache()
    gpu_time, gpu_accuracy = train_model(gpu_device, epochs=20)
    
    # Show speedup
    speedup = cpu_time / gpu_time
    print(f"\n=== Comparison Results ===")
    print(f"CPU time: {cpu_time:.2f} seconds")
    print(f"GPU time: {gpu_time:.2f} seconds")
    print(f"Speedup: {speedup:.2f}x")
    print(f"CPU accuracy: {cpu_accuracy:.4f}")
    print(f"GPU accuracy: {gpu_accuracy:.4f}")


GPU available: NVIDIA GeForce RTX 5060 Ti

=== Training on cpu ===
Loading and preparing data...
Model created with 737818 parameters
Starting training...
Epoch 1/20 - Train Loss: 1.8322, Test Loss: 0.9740, Accuracy: 0.7040
Epoch 2/20 - Train Loss: 0.9238, Test Loss: 0.6569, Accuracy: 0.8025
Epoch 3/20 - Train Loss: 0.7504, Test Loss: 0.6305, Accuracy: 0.8173
Epoch 4/20 - Train Loss: 0.7082, Test Loss: 0.5420, Accuracy: 0.8423
Epoch 5/20 - Train Loss: 0.6509, Test Loss: 0.5089, Accuracy: 0.8470
Epoch 6/20 - Train Loss: 0.6781, Test Loss: 0.4352, Accuracy: 0.8820
Epoch 7/20 - Train Loss: 0.6483, Test Loss: 0.4154, Accuracy: 0.8780
Epoch 8/20 - Train Loss: 0.5851, Test Loss: 0.3578, Accuracy: 0.8890
Epoch 9/20 - Train Loss: 0.5577, Test Loss: 0.3852, Accuracy: 0.8818
Epoch 10/20 - Train Loss: 0.5508, Test Loss: 0.3655, Accuracy: 0.8978
Epoch 11/20 - Train Loss: 0.5656, Test Loss: 0.3720, Accuracy: 0.8950
Epoch 12/20 - Train Loss: 0.5971, Test Loss: 0.3539, Accuracy: 0.9025
Epoch 13/20 - 